# 08 — Pattern Matching

Notebook 07 gave you case classes and enums — the way Scala expresses data. This notebook gives you the partner operation: **pattern matching** — the way you take that data apart again and react to its shape.

If you have come from a C-family language, the closest cousin is `switch`. Scala's `match` is far more capable. It can match on literal values like `switch`, but it can also match on types, deconstruct case classes, bind sub-values to names, apply boolean guards, and — most importantly — get checked for **exhaustiveness** by the compiler so you cannot accidentally leave a case unhandled.

## The shape of a `match` expression

A `match` evaluates the scrutinee (the expression on the left of `match`) once and tries each `case` from top to bottom. The first matching case wins; its right-hand side becomes the value of the whole expression.

In [ ]:
val n = 3

val label = n match
  case 1 => "one"
  case 2 => "two"
  case 3 => "three"
  case _ => "something else"

// label: String = "three"

Five things to notice in that small example:

- `match` is an **expression** — it returns a value. Same mindset as notebook 02.
- Each `case` line names a *pattern* on the left of `=>` and a result expression on the right.
- The result type is the *least upper bound* of all the right-hand sides — here, `String`.
- `_` is the **wildcard pattern** — it matches anything. Used as the last case it acts like a default.
- The first matching case wins. Order matters — list more specific patterns before more general ones.

## The seven patterns you'll use daily

Patterns compose, but they're built from a small fixed alphabet. Here's the whole vocabulary in one place:

```
  case 42                 <- literal pattern
  case _                  <- wildcard pattern
  case x                  <- variable pattern (binds the value to name x)
  case _: String          <- type pattern
  case (a, b)             <- tuple pattern
  case User(n, e, _)      <- constructor pattern (case class / enum)
  case h :: t             <- sequence pattern (cons cell)
```

We'll see each one in context below.

## Literal and wildcard

The two simplest. A literal pattern matches exactly that value, by `==`. A wildcard pattern matches anything and binds nothing.

In [ ]:
def describe(x: Int): String = x match
  case 0  => "zero"
  case 1  => "one"
  case -1 => "minus one"
  case _  => "something else"

## Variable pattern — bind what you match

A lowercase identifier in pattern position **binds** the value to that name for the right-hand side. This is the difference between a wildcard (matches and forgets) and a variable pattern (matches and names).

In [ ]:
def describe(x: Int): String = x match
  case 0          => "zero"
  case n if n < 0 => s"negative: $n"   // n bound; reused in the body
  case n          => s"positive: $n"

describe(-3)    // "negative: -3"
describe(5)     // "positive: 5"

Two gotchas about variable patterns:

- **Lowercase** identifier binds. `case n => ...` binds the matched value to `n`.
- **Uppercase** identifier is treated as a *stable reference* to an existing value — it does **not** bind. `case None => ...` matches the singleton `None`, it does not bind a fresh variable called `None`. This is why constructor names are PascalCase.

If you ever need to match against a *local lowercase* `val`, surround it in backticks: `` case `expected` => ``.

## Type patterns

`case x: T` matches if the scrutinee is an instance of type `T`, and binds it at that type so you can use it as a `T` on the right-hand side.

In [ ]:
def describe(any: Any): String = any match
  case s: String => s"a string of length ${s.length}"
  case n: Int    => s"the number $n"
  case xs: List[_] => s"a list with ${xs.size} elements"
  case _         => "something else"

describe("scala")     // "a string of length 5"
describe(42)          // "the number 42"
describe(List(1,2,3)) // "a list with 3 elements"

Type patterns are the safe replacement for the `instanceof` plus cast pattern from Java. The check and the cast happen together, and the compiler enforces that you use the bound name at the right type.

**Caveat: type erasure.** On the JVM, generic type parameters are erased at runtime — `List[Int]` and `List[String]` look identical. So `case xs: List[Int]` actually checks only that it's a `List`. The compiler will warn you. Use `List[_]` to match any list, and if you need to know the element type, design your code so the static types carry that information.

## Tuple patterns

A tuple pattern deconstructs a tuple positionally, binding each component to a name.

In [ ]:
def diagonalKind(p: (Int, Int)): String = p match
  case (0, 0)         => "origin"
  case (x, y) if x == y  => s"on main diagonal at $x"
  case (x, y) if x == -y => s"on anti-diagonal at $x"
  case (x, y)         => s"point ($x, $y)"

diagonalKind((3, 3))    // "on main diagonal at 3"
diagonalKind((2, -2))   // "on anti-diagonal at 2"

## Constructor patterns — deconstructing case classes and enums

This is what makes pattern matching shine. A constructor pattern matches a case class or an enum case and binds each field to a name.

In [ ]:
enum Shape:
  case Circle(radius: Double)
  case Rectangle(width: Double, height: Double)
  case Triangle(base: Double, height: Double)

def area(s: Shape): Double = s match
  case Shape.Circle(r)        => math.Pi * r * r
  case Shape.Rectangle(w, h)  => w * h
  case Shape.Triangle(b, h)   => 0.5 * b * h

area(Shape.Circle(2.0))         // 12.566...
area(Shape.Rectangle(3.0, 4.0)) // 12.0

Under the hood this works because every case class generates an `unapply` method on its companion object. Pattern matching calls `unapply`, gets the fields back as a tuple, and binds them to the names you wrote. This is also how you can write your own custom *extractors* — but in practice you almost never need to; case classes and enums cover the common case completely.

Patterns also **nest**. The fields you bind can themselves be patterns:

In [ ]:
case class Order(id: Int, customer: String, total: Double)

def summarise(o: Order): String = o match
  case Order(_, "alice", t)   => s"alice spent $t"
  case Order(_, name, 0.0)    => s"$name placed an empty order"
  case Order(id, name, t)     => s"order #$id by $name: $t"

Three patterns in one signature: the first matches when the customer is the literal string `"alice"`, the second when the total is `0.0`, the third catches anything else. The wildcard `_` ignores fields you don't care about. The compiler tries them top-to-bottom; the first match wins.

## Guards — boolean refinement

Append `if` plus a boolean expression to a case to add an extra runtime condition. The case fires only if the pattern matches *and* the guard is true.

In [ ]:
def classify(n: Int): String = n match
  case 0           => "zero"
  case x if x < 0  => "negative"
  case x if x < 10 => "small"
  case _           => "big"

Guards are useful when a condition isn't expressible as a pure pattern — anything involving comparisons, multi-field constraints, or external state. Don't over-use them: if you can refactor the data so the case naturally distinguishes itself, you usually should.

## Alternatives — `|`

Use `|` to share a right-hand side across several patterns. Caveat: alternatives **cannot bind variables** (the names would have to agree across alternatives), only check shape.

In [ ]:
def isVowel(c: Char): Boolean = c match
  case 'a' | 'e' | 'i' | 'o' | 'u' => true
  case _                            => false

## Sequence patterns — `::` and `List(...)`

Lists have two pattern shapes you'll see constantly. `head :: tail` matches a non-empty list by splitting off the first element. `Nil` matches the empty list. `List(a, b, c)` matches a list of exactly three elements.

In [ ]:
def describe(xs: List[Int]): String = xs match
  case Nil               => "empty"
  case x :: Nil          => s"one: $x"
  case x :: y :: Nil     => s"two: $x, $y"
  case head :: rest      => s"head=$head, rest=$rest"

describe(Nil)             // "empty"
describe(List(7))         // "one: 7"
describe(List(1, 2, 3))   // "head=1, rest=List(2, 3)"

This is the idiomatic way to write recursive functions on lists. The cons pattern (`x :: rest`) gives you the head and the tail in one line, ready to recurse on `rest`.

## The `@` binding — name a sub-pattern

Sometimes you want to deconstruct *and* keep the whole matched value under a name. Use the at-sign binding: `name @ pattern`.

In [ ]:
enum Tree:
  case Leaf
  case Node(value: Int, left: Tree, right: Tree)

def describe(t: Tree): String = t match
  case Tree.Leaf                                => "leaf"
  case full @ Tree.Node(_, Tree.Leaf, Tree.Leaf) => s"singleton: $full"
  case Tree.Node(v, _, _)                       => s"node holding $v"

Inside the right-hand side of the second case, `full` refers to the entire `Tree.Node(...)` value while the inner pattern is also enforced. Without the `@`, you'd have to repeat the whole node literal in the body. With it, you get to deconstruct and keep the whole at once.

## Exhaustiveness — the compiler has your back

This is where pattern matching becomes a *design tool*, not just a syntax convenience.

When the scrutinee's type is a *sealed* hierarchy — every case class or every enum case is one — the compiler can enumerate the alternatives. If your `match` doesn't cover them all, you get a warning. Treat that warning as an error: it means your code silently doesn't handle a case that exists in your data.

In [ ]:
enum PaymentResult:
  case Success(txnId: String)
  case Declined(reason: String)
  case Failed(error: String)

def message(r: PaymentResult): String = r match
  case PaymentResult.Success(id)    => s"ok: $id"
  case PaymentResult.Declined(why)  => s"declined: $why"
  // missing Failed -> warning: match may not be exhaustive

Now imagine you go add a new case — `case Cancelled(orderId: Long)` — to `PaymentResult`. Every `match` in your codebase that didn't include `Cancelled` lights up with the same warning. The compiler hands you the worklist. This is what makes algebraic data types plus pattern matching such a powerful pairing: **the type system becomes the change-impact analysis tool.**

## Pattern matching outside `match`

`match` is the explicit form, but patterns also work in two other places.

**On the left side of a `val`** — to destructure a value at the binding site.

In [ ]:
val (name, age) = ("alice", 30)
// name: String = "alice"
// age: Int = 30

case class User(name: String, email: String, age: Int)
val User(uname, _, uage) = User("ganesh", "g@example.com", 35)
// uname: String = "ganesh"
// uage: Int = 35

**Inside a `for` comprehension** — both in the generator and on the left of `<-`.

In [ ]:
val ages = Map("alice" -> 30, "bob" -> 25, "cara" -> 41)

for (name, age) <- ages if age >= 30 do
  println(s"$name is $age")
// alice is 30
// cara is 41

Both forms quietly rely on the same `unapply` machinery as `match`. Once you internalise patterns, the same shapes work everywhere a value is bound.

## Putting it together

A small but realistic example: classify trading events, using nested patterns, guards, and exhaustiveness.

In [ ]:
enum Event:
  case Trade(symbol: String, price: Double, quantity: Int)
  case Quote(symbol: String, bid: Double, ask: Double)
  case Heartbeat(timestamp: Long)

def alert(e: Event): String = e match
  case Event.Trade(_, _, q) if q > 10_000        => "large trade!"
  case Event.Trade(sym, price, q)                => s"trade $sym @ $price x $q"
  case Event.Quote(sym, bid, ask) if ask - bid > 1.0 => s"wide spread on $sym"
  case Event.Quote(sym, _, _)                    => s"quote $sym"
  case Event.Heartbeat(_)                        => "heartbeat"

alert(Event.Trade("AAPL", 188.0, 50_000))      // "large trade!"
alert(Event.Quote("AAPL", 187.5, 189.0))       // "wide spread on AAPL"

Three things working together in that example:

- **Constructor patterns** deconstruct each event case.
- **Guards** refine on quantity and on spread width.
- **Exhaustiveness** — try deleting the `Heartbeat` case and the compiler warns you immediately.

Add a new `case Cancel(orderId: Long)` to `Event` and the `alert` function lights up as non-exhaustive. The type system is telling you exactly where to think.

## What's next

Notebook 09 puts pattern matching to immediate work on `Option`, `Try`, and `Either` — the three standard-library types that let you handle absence, failure, and error union without resorting to `null` or thrown exceptions. The `Some(x) | None` shape from this notebook is exactly how you'll deconstruct an `Option`.